### 题目代码

In [1]:
import math
 
 

def is_prime(num): 
    if num < 2: 
        return False
    for x in range(2, int(num ** 0.5) + 1):
        if num % x == 0:
            return False
    return True
 
 
def found_nums(lst, length, s, k, m):  
    """
    使用深度搜索，结合回溯算法
    lst: 当前数字列表
    length: 当前数字长度
    s: 当前数字和
    k: 目标数字长度
    m: 目标数字和
    """
    # 提前剪枝
    if s + (k - length) * 9 < m:  
        return
    
    # 迭代出口
    if length == k:  
        if s == m:  
            res = ''.join(lst)  
            n = sum(int(d) for d in str(int(res) + 1)) 
            p = math.gcd(n, m) 
            if p > 2 and is_prime(p):  
                ans.append(f"{n} {res}")  
        return
    
    # 核心迭代递归代码
    for t in range(10): 
        if s + t <= m: # 剪枝
            tmp = lst[length] # 记录当前位的值，便于回溯
            lst[length] = str(t) # 设置当前位
            found_nums(lst, length + 1, s + t, k, m) # 递归调用，深入下一位
            lst[length] = tmp # 回溯
    return
 
 

# # 获取首行输入数据4
# N = int(input())
 
# # 计算结果并输出
# for i in range(N):
#     print(f"Case {i+1}")
#     K, M = map(int, input().split())
#     ans = []
#     a = ['0'] * K
#     for j in range(1, 10):
#         a[0] = str(j)
#         found_nums(a, 1, j, K, M)

#     ans.sort(key=lambda x: int(x.split()[0]))

#     print('\n'.join(ans) if ans else 'No Solution')


### 绘制搜索树函数

In [2]:
import math
from typing import List, Optional

class Node:
    __slots__ = ("digit","s","depth","children","is_leaf","ok")
    def __init__(self,digit:Optional[int],s:int,depth:int):
        self.digit = digit      # 根节点为 None
        self.s = s              # 累计和
        self.depth = depth      # 已放置位数
        self.children: List['Node'] = []
        self.is_leaf = False
        self.ok = False         # 是否满足 gcd(sum_digits(num+1), M) 是素数且 >2

    def label(self):
        base = "root" if self.digit is None else str(self.digit)
        tag = " [OK]" if self.ok else ""
        return f"{base}(s={self.s},d={self.depth}){tag}"


def build_tree(K:int,M:int,max_nodes:int=50000)->Node:
    global_nodes = 0
    root = Node(None,0,0)

    def sum_digits(x:int)->int:
        return sum(int(c) for c in str(x))

    def expand(node:Node,prefix_digits:List[int]):
        nonlocal global_nodes
        # 剪枝：最大理论可加和不足
        if node.s + (K - node.depth)*9 < M:
            return
        if node.depth == K:
            if node.s == M:
                node.is_leaf = True
                # 计算 number+1 的数字和
                number = 0
                for d in prefix_digits:
                    number = number*10 + d
                n = sum_digits(number+1)
                g = math.gcd(n,M)
                if g > 2 and is_prime(g):
                    node.ok = True
            return
        start = 1 if node.depth==0 else 0
        for t in range(start,10):
            if node.s + t <= M:
                child = Node(t,node.s + t,node.depth+1)
                node.children.append(child)
                global_nodes += 1
                if global_nodes >= max_nodes:
                    return
                prefix_digits.append(t)
                expand(child,prefix_digits)
                prefix_digits.pop()

    expand(root,[])
    return root


def render_ascii(root:Node,max_leaves:int=5000):
    lines: List[str] = []
    leaves = 0
    def dfs(node:Node,prefix:str,is_last:bool):
        nonlocal leaves
        connector = "└─ " if is_last else "├─ "
        if node.digit is None:
            lines.append(node.label())
        else:
            lines.append(prefix + connector + node.label())
        if node.is_leaf:
            leaves += 1
            if leaves > max_leaves:
                lines.append(f"... 叶子超过上限 {max_leaves}，停止渲染 ...")
                return
        if node.children and leaves <= max_leaves:
            new_prefix = prefix + ("   " if is_last else "│  ")
            for i,ch in enumerate(node.children):
                if leaves > max_leaves:
                    break
                dfs(ch,new_prefix,i==len(node.children)-1)
    dfs(root,"",True)
    return lines


def print_search_tree(K:int,M:int,max_nodes:int=50000,max_leaves:int=5000):
    root = build_tree(K,M,max_nodes=max_nodes)
    for line in render_ascii(root,max_leaves=max_leaves):
        print(line)


### 调用函数

In [3]:
# 示例：打印 K=3, M=6 的搜索树（可自行修改参数）
print_search_tree(K=3, M=6, max_nodes=20000, max_leaves=3000)

root(s=0,d=0)
   ├─ 1(s=1,d=1)
   │  ├─ 0(s=1,d=2)
   │  │  ├─ 0(s=1,d=3)
   │  │  ├─ 1(s=2,d=3)
   │  │  ├─ 2(s=3,d=3)
   │  │  ├─ 3(s=4,d=3)
   │  │  ├─ 4(s=5,d=3)
   │  │  └─ 5(s=6,d=3)
   │  ├─ 1(s=2,d=2)
   │  │  ├─ 0(s=2,d=3)
   │  │  ├─ 1(s=3,d=3)
   │  │  ├─ 2(s=4,d=3)
   │  │  ├─ 3(s=5,d=3)
   │  │  └─ 4(s=6,d=3)
   │  ├─ 2(s=3,d=2)
   │  │  ├─ 0(s=3,d=3)
   │  │  ├─ 1(s=4,d=3)
   │  │  ├─ 2(s=5,d=3)
   │  │  └─ 3(s=6,d=3)
   │  ├─ 3(s=4,d=2)
   │  │  ├─ 0(s=4,d=3)
   │  │  ├─ 1(s=5,d=3)
   │  │  └─ 2(s=6,d=3)
   │  ├─ 4(s=5,d=2)
   │  │  ├─ 0(s=5,d=3)
   │  │  └─ 1(s=6,d=3)
   │  └─ 5(s=6,d=2)
   │     └─ 0(s=6,d=3)
   ├─ 2(s=2,d=1)
   │  ├─ 0(s=2,d=2)
   │  │  ├─ 0(s=2,d=3)
   │  │  ├─ 1(s=3,d=3)
   │  │  ├─ 2(s=4,d=3)
   │  │  ├─ 3(s=5,d=3)
   │  │  └─ 4(s=6,d=3)
   │  ├─ 1(s=3,d=2)
   │  │  ├─ 0(s=3,d=3)
   │  │  ├─ 1(s=4,d=3)
   │  │  ├─ 2(s=5,d=3)
   │  │  └─ 3(s=6,d=3)
   │  ├─ 2(s=4,d=2)
   │  │  ├─ 0(s=4,d=3)
   │  │  ├─ 1(s=5,d=3)
   │  │  └─ 2(s=6,d=3)
   │  ├─ 3(s=